# Tree diagnostics: is the supermatrix any good?

This notebook looks at the **concatenated alignment** (the supermatrix the tree is built from), before
any tree exists. It is executed by the `tree_diagnostics` rule, once per tree variant:

| label | meaning |
|---|---|
| `all` / `derep` | every genome, or only the dereplicated representatives (external genomes always kept) |
| `single` / `multi` | Complete BUSCO genes only, or also the top-scoring copy of Duplicated genes |
| `subset ...` | a quick-check subset (`tree.subset`), not the full genome set |

The label of this run is printed in the first cell. **Do not compare numbers between a subset run and a
full run**: sampling changes the clades, the distances and the RED cutoffs.

Questions answered, in order:

1. **Occupancy** - does every genome have enough genes, or are some mostly gaps?
2. **Gene overlap and divergence** - are genomes of the same clade more similar, and do they share more genes, than genomes of different clades?
3. **Per-gene separation** - does any single gene fail to tell clades apart?
4. **Multicopy genes** - is "take the top-scoring copy" a good rule?

Clade fragmentation on the finished tree is in the `tree_accuracy` notebook.

## 0. Setup

Parameters are written by the `tree_diagnostics` rule into a JSON file (path in the `NB_PARAMS`
environment variable), so the notebook has no hard-coded paths. Statistics live in
`scripts/tree_diagnostics_lib.py` (unit-tested); this notebook loads data, calls them and shows results.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# Parameters written by the Snakemake rule
PARAMS = json.load(open(os.environ["NB_PARAMS"]))
sys.path.insert(0, PARAMS["scripts_dir"])
import tree_diagnostics_lib as lib

OUT = Path(PARAMS["out_dir"])
OUT.mkdir(parents=True, exist_ok=True)
RANKS = ["phylum", "class", "order", "family", "genus"]
N_PAIRS = PARAMS["n_pairs"]  # genome pairs sampled per rank

print("Tree variant:", PARAMS["label"])
print("Alignment   :", PARAMS["alignment"])

In [ ]:
def show_all(df, width=200):
    '''
    Prints an entire polars dataframe in the notebook output.

    Parameters
    ----------
    df : pl.DataFrame
        The dataframe to be printed.
    width : int, optional
        The width (in characters) of the printed dataframe.
        Defaults to 200.
    '''
    with pl.Config() as cfg:
        cfg.set_tbl_cols(-1)
        cfg.set_tbl_rows(-1)
        cfg.set_tbl_width_chars(width)
        cfg.set_fmt_str_lengths(width)
        print(df)

## 1. Load the alignment

* `ids` / `matrix` - one row per genome, one column per aligned position (`-` = gap).
* `partitions` - which columns belong to which BUSCO gene.
* `taxonomy` - the database lineage of each genome (`conservative_taxonomy.tsv`: complete **and** partial lineages).

In [ ]:
ids, matrix = lib.load_alignment(PARAMS["alignment"])
partitions = lib.read_partitions(PARAMS["partitions"])
taxonomy = lib.read_taxonomy(PARAMS["taxonomy"])

n_labelled = sum(1 for genome in ids if genome in taxonomy)
print(f"{len(ids)} genomes x {matrix.shape[1]} columns, {len(partitions)} genes")
print(f"{n_labelled} genomes have a database taxonomy; the other {len(ids) - n_labelled} are unlabelled (e.g. MAGs)")

## 2. Occupancy: genes per genome and gaps

For each genome: the number of genes with at least one residue, and the fraction of its row that is gap.

**How to read it.** Most rows should have well over half of the genes. A long tail of genomes with few genes
adds mostly gaps to the supermatrix; they tend to be placed unstably and can look "misplaced". The workflow
already drops genomes below `alignment.min_genes_per_genome`; if many rows are still above ~90% gaps,
raise that threshold or check the genomes' BUSCO completeness.

In [ ]:
occupancy, present = lib.occupancy(ids, matrix, partitions)
occupancy = pl.DataFrame(occupancy)  # columns: genome, genes, gap_fraction

summary = occupancy.select(
    pl.len().alias("genomes"),
    pl.col("genes").median().alias("median_genes"),
    pl.col("genes").min().alias("min_genes"),
    pl.col("genes").quantile(0.1).alias("genes_10th_pct"),
    pl.col("gap_fraction").median().alias("median_gap_fraction"),
    (pl.col("gap_fraction") >= 0.9).sum().alias("rows_90pct_gaps"),
)
show_all(summary)
occupancy.write_csv(OUT / "occupancy.tsv", separator="\t")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].hist(occupancy["genes"], bins=40)
axes[0].set_title("genes per genome")
axes[1].hist(occupancy["gap_fraction"], bins=40)
axes[1].set_title("gap fraction per genome row")
axes[2].hist(present.sum(axis=0), bins=40)
axes[2].set_title("genomes per gene")
plt.show()

The genomes with the fewest genes - the first candidates to inspect or drop:

In [ ]:
labelled = pl.DataFrame({
    "genome": ids,
    "phylum": [taxonomy.get(genome, {}).get("phylum", "-") for genome in ids],
})
show_all(occupancy.sort("genes").head(15).join(labelled, on="genome", how="left"))

## 3. Do clades share genes, and are they closer in sequence?

Pairs of genomes are sampled at each rank:

* **within** - both genomes carry the same clade name at that rank,
* **between** - they carry different names.

For each pair we compute

* `pdist` - proportion of differing residues over the columns where both genomes have a residue,
* `jaccard` - shared genes / genes in either genome,
* `shared_genes` - number of genes both have.

`separation_pdist` is the probability that a *within* pair is closer than a *between* pair
(1.0 = clades are perfectly distinguishable, 0.5 = no information, below 0.5 = inverted).

**How to read it.**

* `median_pdist_within` should fall steadily from phylum to genus, while `median_pdist_between` stays
  high. Once between-clade distance stops growing with depth, p-distance is saturated and cannot rank deep
  divergences - the tree then relies on the model.
* `separation_pdist` is high at every rank when the *alignment* distinguishes the clades. If the tree later
  fragments a clade even though separation is high, suspect the labels, a few misplaced genomes or the
  tree model rather than the data.
* `jaccard` within > between means clade members have similar gene sets. That can be real, but it can also
  mean *missing* genes follow the clade, so genomes could group by what they lack. Look at it together
  with the occupancy section.

In [ ]:
report = lib.rank_report(ids, matrix, present, partitions, taxonomy,
                         ranks=RANKS, n_pairs=N_PAIRS, seed=1)
rank_summary = pl.DataFrame([report[rank]["summary"] for rank in report])
show_all(rank_summary.select(
    "rank", "n_within", "n_between",
    pl.col("median_pdist_within").round(3), pl.col("median_pdist_between").round(3),
    pl.col("median_jaccard_within").round(3), pl.col("median_jaccard_between").round(3),
    pl.col("median_shared_genes_within"), pl.col("median_shared_genes_between"),
    pl.col("separation_pdist").round(3),
))
rank_summary.write_csv(OUT / "rank_summary.tsv", separator="\t")

In [ ]:
def boxplot_within_between(axis, report, key, title):
    '''
    Draws within-clade and between-clade values of one pair statistic for every rank.

    Parameters
    ----------
    axis : matplotlib.axes.Axes
        Axis to draw on.
    report : dict
        Output of lib.rank_report.
    key : str
        Pair statistic: "pdist", "jaccard" or "shared_genes".
    title : str
        Axis title.
    '''
    data, labels = [], []
    for rank, result in report.items():
        for kind in ("within", "between"):
            values = np.asarray(result[kind][key], dtype=float)
            data.append(values[~np.isnan(values)])
            labels.append(f"{rank[:4]}\n{kind[:3]}")
    axis.boxplot(data, showfliers=False)
    axis.set_xticks(range(1, len(labels) + 1))
    axis.set_xticklabels(labels, fontsize=7)
    axis.set_title(title)


fig, axes = plt.subplots(1, 3, figsize=(16, 4))
boxplot_within_between(axes[0], report, "pdist", "p-distance (shared sites)")
boxplot_within_between(axes[1], report, "jaccard", "gene-set Jaccard")
boxplot_within_between(axes[2], report, "shared_genes", "shared genes")
plt.show()

## 4. Per-gene separation

The same within/between comparison, one gene at a time. A gene whose within-clade distances are not smaller
than its between-clade distances adds noise (saturated, badly aligned or paralogous), and is a candidate
to drop.

Only genes with at least `MIN_PAIRS` within-clade pairs that have data are summarised, because a gene
seen in few sampled pairs gives a noisy value.

**How to read it.** A handful of genes slightly below the rest is normal. A clear low tail (for example
several genes under ~0.6) suggests removing those genes and re-running; a uniformly low distribution points at
the labels or the alignment step instead.

In [ ]:
MIN_PAIRS = 30

gene_rows = []
for rank in report:
    for gene, separation, n_pairs in lib.gene_separation(report, partitions, rank):
        gene_rows.append({"rank": rank, "gene": gene, "separation": separation, "within_pairs_with_data": n_pairs})
gene_separation = pl.DataFrame(gene_rows)
gene_separation.write_csv(OUT / "gene_separation.tsv", separator="\t")

usable = gene_separation.filter(
    (pl.col("within_pairs_with_data") >= MIN_PAIRS) & pl.col("separation").is_not_nan()
)
per_rank = usable.group_by("rank").agg(
    pl.len().alias("genes"),
    pl.col("separation").median().round(3).alias("median_separation"),
    pl.col("separation").min().round(3).alias("min_separation"),
    (pl.col("separation") < 0.6).sum().alias("genes_below_0.6"),
)
show_all(per_rank.sort("rank"))

In [ ]:
print("Lowest-separation genes per rank:")
for rank in RANKS:
    worst = usable.filter(pl.col("rank") == rank).sort("separation").head(6)
    print(f"  {rank:7}", ", ".join(f"{gene}({value:.2f})" for gene, value in zip(worst["gene"], worst["separation"])))

fig, axis = plt.subplots(figsize=(7, 3.5))
for rank in RANKS:
    axis.hist(usable.filter(pl.col("rank") == rank)["separation"], bins=25, histtype="step", label=rank)
axis.set_xlabel("per-gene separation")
axis.legend()
plt.show()

## 5. Multicopy genes: is the top-scoring copy a good choice?

When a BUSCO gene is *Duplicated* the aggregation keeps only the copy with the highest BUSCO score. That is
a heuristic: the best-scoring copy is not necessarily the orthologous one (a paralog can score higher).

Test: for randomly chosen Duplicated genes, compare each copy with the **same gene in related genomes**
(same family, else order, class, phylum; up to 25 peers) using 3-mer similarity as a cheap proxy for
orthology. We then ask whether the chosen (top-score) copy is also the copy most similar to the peers.

**How to read it.**

* `agree` - the share of genes where both choices are the same copy. High = the heuristic is fine.
* `similarity_gap` - when they disagree, how much more peer-like the better copy is (3-mer Jaccard).
  A large gap means the heuristic is picking a clearly worse copy.
* `score_ratio` - runner-up score / top score. Near 1 means the two copies are nearly tied, so the score
  carries little information and a similarity-based choice would be more reliable.

If `agree` is much below ~90% *and* the gaps are large, replace the rule with "choose the copy closest to
the clade" (or build the gene tree and pick the orthologous copy).

In [ ]:
# BUSCO full tables of every genome: (accession, path)
tables = []
for entry in PARAMS["busco_manifests"]:
    manifest, base = Path(entry["manifest"]), Path(entry["base"])
    if not manifest.exists():
        print("missing manifest:", manifest)
        continue
    qc = pl.read_csv(manifest, separator="\t")
    for accession, result in zip(qc["genome_acc"], qc["busco_result"]):
        path = Path(result) if Path(result).is_absolute() else base / result
        tables.append((accession, str(path)))
print(f"{len(tables)} BUSCO tables")

cases = lib.multicopy_analysis(tables, taxonomy, PARAMS["merged_dir"],
                               max_cases=PARAMS["max_multicopy_cases"], seed=2) if tables else []
print(f"{len(cases)} Duplicated genes evaluated")

In [ ]:
if cases:
    multicopy = pl.DataFrame([
        {k: case[k] for k in ("genome", "gene", "n_copies", "agree", "similarity_gap",
                              "score_ratio", "copy_identity", "peer_rank", "n_peers")}
        for case in cases
    ])
    multicopy.write_csv(OUT / "multicopy_cases.tsv", separator="\t")

    disagree = multicopy.filter(~pl.col("agree"))
    show_all(multicopy.select(
        pl.col("agree").mean().round(3).alias("top_copy_is_most_peer_like"),
        pl.col("score_ratio").median().round(3).alias("median_score_ratio"),
        (pl.col("score_ratio") > 0.9).mean().round(3).alias("share_near_tied"),
        pl.lit(disagree["similarity_gap"].median() if disagree.height else None).alias("median_gap_when_disagree"),
    ))
    show_all(multicopy.group_by("peer_rank").len().sort("len", descending=True))
else:
    print("No multicopy cases (is use_multi_copy enabled, and do genomes have Duplicated genes?)")

In [ ]:
if cases:
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    axes[0].hist(multicopy["similarity_gap"], bins=30)
    axes[0].set_title("peer-similarity gap (best peer-like copy - chosen copy)")
    axes[1].scatter(multicopy["score_ratio"], multicopy["similarity_gap"], s=6)
    axes[1].set_xlabel("score ratio (runner-up / top)")
    axes[1].set_ylabel("gap")
    axes[2].hist(multicopy["copy_identity"], bins=30)
    axes[2].set_title("similarity between the two best copies")
    plt.show()

## 6. Summary

The numbers worth carrying into the other notebooks. Tables written next to this notebook:
`occupancy.tsv`, `rank_summary.tsv`, `gene_separation.tsv`, `multicopy_cases.tsv`.

In [ ]:
print(f"Variant: {PARAMS['label']}")
print(f"Genomes: {len(ids)}   genes: {len(partitions)}   median genes/genome: {int(occupancy['genes'].median())}")
print(f"Rows >= 90% gaps: {int((occupancy['gap_fraction'] >= 0.9).sum())}")
for row in rank_summary.iter_rows(named=True):
    print(f"  {row['rank']:7} separation {row['separation_pdist']:.3f}   "
          f"within/between p-distance {row['median_pdist_within']:.2f}/{row['median_pdist_between']:.2f}")
if cases:
    print(f"Multicopy: top-score copy is the most peer-like in {multicopy['agree'].mean():.0%} of {len(cases)} genes")